# FLUX.1-lite-8B on AWS Neuron

This notebook demonstrates how to compile and run **FLUX.1-lite-8B** (Freepik/flux.1-lite-8B) on trn2.3xlarge using `torch_neuronx.trace()`.

FLUX.1-lite is an 8-billion parameter text-to-image model based on the FLUX architecture with 8 MMDiT double-stream blocks and 38 single-stream DiT blocks. The full transformer exceeds the Neuron compiler's 5M instruction limit (8.6M actual), requiring a **3-part split** strategy.

### Key contribution

This is a complete port of a FLUX-architecture model to AWS Neuron using `torch_neuronx.trace()`, demonstrating:
- **3-part transformer splitting** to work around the 5M instruction limit
- **Staged model loading** with subprocess text encoding to manage HBM
- **Critical finding**: the `--model-type=transformer` compiler flag destroys image quality for FLUX attention patterns

### Model overview

| Property | Value |
|----------|-------|
| Model | FLUX.1-lite-8B (Freepik/flux.1-lite-8B) |
| Architecture | FLUX (MMDiT + DiT) |
| Parameters | ~8B (transformer only) |
| MMDiT blocks | 8 double-stream |
| DiT blocks | 38 single-stream |
| Text encoders | CLIP + T5-XXL |
| Resolution | 1024x1024 |
| Steps | 28 |
| Guidance | 3.5 |

### What this notebook covers

1. Environment setup and model download
2. Architecture overview and split strategy
3. Compilation of all 6 components (CLIP, T5, transformer x3, VAE)
4. End-to-end inference with staged model loading
5. Throughput benchmark
6. Results summary

### Platform

- **Target**: trn2.3xlarge (4 NeuronCores, 96 GB HBM)
- **SDK**: Neuron SDK 2.29
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260410
- **Venv**: `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/`

## Instance Setup

This notebook is designed to run on a **trn2.3xlarge** with the **Deep Learning AMI Neuron (Ubuntu 24.04)**.

To launch your instance, use the AWS Console or CLI. For a walkthrough of launching a Neuron instance, see this video tutorial (starts at the instance launch section):

> **Video Guide**: [Launching a Neuron Instance](https://youtu.be/CyTCTuq1z0Q?t=657)

## Jupyter Kernel Setup

This notebook requires a Python kernel from the pre-installed Neuron virtual environment. There are two ways to set this up:

### Option A: Jupyter Server on the Instance

SSH into your instance and start Jupyter from the Neuron virtual environment:

```bash
source /opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/bin/activate
pip install jupyter
jupyter notebook --no-browser --port=8888
```

Then use SSH port forwarding to access it from your local browser:

```bash
ssh -i "/path/to/sshkey.pem" -L 8888:localhost:8888 ubuntu@<instance_ip>
```

### Option B: VS Code Remote-SSH

With Visual Studio Code installed on your local machine, you can use Remote-SSH to edit and run notebooks directly on the Neuron instance:

1. Select **Remote-SSH: Connect to Host...** from the Command Palette (`F1` or `Shift+Cmd+P`)
2. Enter the full connection string: `ssh -i "/path/to/sshkey.pem" ubuntu@<instance_ip>`
3. VS Code will connect and set up the VS Code server automatically
4. When prompted, browse to your working directory on the instance
5. Some menu commands may appear greyed out, but keyboard shortcuts still work (`Cmd+S` to save, `Ctrl+Shift+\`` for terminal). You may need to restart VS Code.

To use the pre-installed Neuron virtual environment as your Jupyter kernel in VS Code, open a terminal and create a symbolic link:

```bash
ln -s /opt/aws_neuronx_venv_pytorch_2_9_nxd_inference ~/.venv
```

Then select the `.venv` Python interpreter when choosing a kernel for the notebook.

## 1. Environment Setup

In [ ]:
# Verify Neuron devices
!neuron-ls

In [ ]:
# Install dependencies not in the DLAMI
!pip install -q diffusers sentencepiece protobuf

In [ ]:
import os
import copy
import time
import gc
import subprocess
import json

import numpy as np
import torch
import torch.nn as nn
import torch_neuronx

print(f"torch_neuronx: {torch_neuronx.__version__}")
print(f"torch: {torch.__version__}")

In [ ]:
# Configuration
MODEL_ID = "Freepik/flux.1-lite-8B"
RESOLUTION = 1024
IMG_SEQ_LEN = (RESOLUTION // 8 // 2) ** 2   # 4096 for 1024x1024
TXT_SEQ_LEN = 512
CLIP_SEQ_LEN = 77
TOTAL_SEQ_LEN = TXT_SEQ_LEN + IMG_SEQ_LEN   # 4608
HIDDEN_DIM = 3072                             # 24 heads * 128 dim
NUM_STEPS = 28
GUIDANCE_SCALE = 3.5

COMPILE_DIR = os.path.expanduser("~/flux_compile")
os.makedirs(COMPILE_DIR, exist_ok=True)
print(f"Compile directory: {COMPILE_DIR}")

## 2. Architecture and Split Strategy

The FLUX.1-lite transformer has **8.6 million instructions** when compiled as a single graph, exceeding Neuron's 5M instruction limit (`NCC_EVRF007`). We split into 3 parts:

| Part | Contents | Instructions | NEFF Size |
|------|----------|-------------|----------|
| **A** | Embeddings + 8 MMDiT double blocks | ~2.5M | ~9 GB |
| **B** | Single-stream blocks 0-18 (19 blocks) | ~2.5M | ~8 GB |
| **C** | Single-stream blocks 19-37 + norm_out + proj_out (19 blocks) | ~2.5M | ~8 GB |

At each denoising step, we execute A -> concat -> B -> C sequentially.

### HBM management

The Neuron runtime does **not** release HBM when a model is deleted within the same process. This means:
- Text encoders (CLIP + T5) must run in a **separate subprocess** on core 0
- Transformer parts load in the main process on cores 1-3
- VAE decoder loads after transformer parts are unloaded (in a new subprocess or after process restart)

### Critical compiler flag warning

**Do NOT use `--model-type=transformer`** for FLUX models. This flag replaces native softmax with a custom NKI kernel that is numerically inaccurate for FLUX's MMDiT/DiT attention patterns. It causes:
- Cosine similarity drops to 0.916 (should be 0.99+)
- Blank/garbage images after 28 denoising steps
- 37% mean error per step

Use `-O1` only (no model-type flags) for correct output.

## 3. Define Wrapper Modules

Each part of the transformer is wrapped in an `nn.Module` for `torch_neuronx.trace()`.

In [ ]:
class FluxPartA(nn.Module):
    """Embeddings + 8 MMDiT double blocks.
    
    Inputs:
      hidden_states:         [B, 4096, 64]
      encoder_hidden_states: [B, 512, 4096]
      pooled_projections:    [B, 768]
      timestep:              [B]
      guidance:              [B]
      rope_cos:              [4608, 128]
      rope_sin:              [4608, 128]
    
    Outputs:
      hidden_states:         [B, 4096, 3072]
      encoder_hidden_states: [B, 512, 3072]
      temb:                  [B, 3072]
    """
    def __init__(self, model):
        super().__init__()
        self.x_embedder = model.x_embedder
        self.time_text_embed = model.time_text_embed
        self.context_embedder = model.context_embedder
        self.transformer_blocks = model.transformer_blocks

    def forward(self, hidden_states, encoder_hidden_states, pooled_projections,
                timestep, guidance, rope_cos, rope_sin):
        image_rotary_emb = (rope_cos, rope_sin)
        hidden_states = self.x_embedder(hidden_states)
        timestep = timestep.to(hidden_states.dtype) * 1000
        guidance = guidance.to(hidden_states.dtype) * 1000
        temb = self.time_text_embed(timestep, guidance, pooled_projections)
        encoder_hidden_states = self.context_embedder(encoder_hidden_states)
        for block in self.transformer_blocks:
            encoder_hidden_states, hidden_states = block(
                hidden_states=hidden_states,
                encoder_hidden_states=encoder_hidden_states,
                temb=temb,
                image_rotary_emb=image_rotary_emb,
            )
        return hidden_states, encoder_hidden_states, temb


class FluxPartB(nn.Module):
    """First 19 single-stream DiT blocks.
    
    Inputs:
      hidden_states: [B, 4608, 3072]  (concatenated encoder + image)
      temb:          [B, 3072]
      rope_cos:      [4608, 128]
      rope_sin:      [4608, 128]
    """
    def __init__(self, blocks):
        super().__init__()
        self.single_transformer_blocks = blocks

    def forward(self, hidden_states, temb, rope_cos, rope_sin):
        image_rotary_emb = (rope_cos, rope_sin)
        for block in self.single_transformer_blocks:
            hidden_states = block(
                hidden_states=hidden_states,
                temb=temb,
                image_rotary_emb=image_rotary_emb,
            )
        return hidden_states


class FluxPartC(nn.Module):
    """Last 19 single-stream DiT blocks + norm_out + proj_out.
    
    Inputs:
      hidden_states: [B, 4608, 3072]
      temb:          [B, 3072]
      rope_cos:      [4608, 128]
      rope_sin:      [4608, 128]
    
    Outputs:
      output: [B, 4096, 64]  (noise prediction for image tokens only)
    """
    def __init__(self, blocks, norm_out, proj_out):
        super().__init__()
        self.single_transformer_blocks = blocks
        self.norm_out = norm_out
        self.proj_out = proj_out

    def forward(self, hidden_states, temb, rope_cos, rope_sin):
        image_rotary_emb = (rope_cos, rope_sin)
        for block in self.single_transformer_blocks:
            hidden_states = block(
                hidden_states=hidden_states,
                temb=temb,
                image_rotary_emb=image_rotary_emb,
            )
        # Remove text tokens, keep image tokens only
        hidden_states = hidden_states[:, TXT_SEQ_LEN:, ...]
        hidden_states = self.norm_out(hidden_states, temb)
        return self.proj_out(hidden_states)

In [ ]:
# Text encoder wrappers

class CLIPWrapper(nn.Module):
    """CLIP text encoder -> pooled output [B, 768]."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids):
        out = self.model(input_ids, output_hidden_states=False, return_dict=False)
        return out[1]  # pooled_output


class T5Wrapper(nn.Module):
    """T5-XXL text encoder -> hidden states [B, 512, 4096]."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids):
        out = self.model(input_ids, output_hidden_states=False, return_dict=False)
        return out[0]  # last_hidden_state

In [ ]:
def precompute_flux_rope(transformer, txt_seq_len=TXT_SEQ_LEN, img_h=64, img_w=64):
    """Precompute RoPE embeddings on CPU for fixed 1024x1024 resolution."""
    txt_ids = torch.zeros(txt_seq_len, 3)
    img_ids = torch.zeros(img_h, img_w, 3)
    img_ids[..., 1] = img_ids[..., 1] + torch.arange(img_h)[:, None]
    img_ids[..., 2] = img_ids[..., 2] + torch.arange(img_w)[None, :]
    img_ids = img_ids.reshape(img_h * img_w, 3)
    ids = torch.cat((txt_ids, img_ids), dim=0)
    with torch.no_grad():
        rope_emb = transformer.pos_embed(ids)
    if isinstance(rope_emb, tuple):
        return rope_emb[0].float(), rope_emb[1].float()
    else:
        return rope_emb.float()


def compile_part(name, wrapper, example_inputs, compiler_args=None):
    """Compile a single component and save the NEFF."""
    workdir = os.path.join(COMPILE_DIR, name)
    neff_path = os.path.join(workdir, "model.pt")
    os.makedirs(workdir, exist_ok=True)
    
    if os.path.exists(neff_path):
        print(f"  [{name}] Loading cached NEFF...")
        return 0.0
    
    if compiler_args is None:
        compiler_args = ["-O1"]
    
    print(f"  [{name}] Compiling...")
    t0 = time.time()
    traced = torch_neuronx.trace(
        wrapper, example_inputs,
        compiler_workdir=workdir,
        compiler_args=compiler_args,
    )
    torch.jit.save(traced, neff_path)
    elapsed = time.time() - t0
    print(f"  [{name}] Compiled in {elapsed:.1f}s ({elapsed / 60:.1f} min)")
    del traced
    gc.collect()
    return elapsed

## 4. Compile Text Encoders

CLIP and T5-XXL are compiled with `-O1` only.

In [ ]:
from diffusers import FluxPipeline

# Load pipeline once for CLIP
print("Loading pipeline for text encoder compilation...")
pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)

# CLIP
clip_wrapper = CLIPWrapper(copy.deepcopy(pipe.text_encoder))
clip_input = torch.zeros(1, CLIP_SEQ_LEN, dtype=torch.long)
t_clip = compile_part("clip", clip_wrapper, clip_input, compiler_args=["-O1"])
del clip_wrapper

# T5
t5_wrapper = T5Wrapper(copy.deepcopy(pipe.text_encoder_2))
t5_input = torch.zeros(1, TXT_SEQ_LEN, dtype=torch.long)
t_t5 = compile_part("t5", t5_wrapper, t5_input, compiler_args=["-O1"])
del t5_wrapper

gc.collect()
print(f"\nText encoders: CLIP={t_clip:.1f}s, T5={t_t5:.1f}s")

## 5. Compile Transformer (3 Parts)

Each part is compiled separately with `-O1`. The full compilation takes ~90 minutes.

In [ ]:
# Reuse pipeline if still loaded, otherwise reload
try:
    _ = pipe.transformer
except:
    print("Reloading pipeline...")
    pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)

transformer = pipe.transformer
print(f"MMDiT blocks: {len(transformer.transformer_blocks)}")
print(f"Single blocks: {len(transformer.single_transformer_blocks)}")

# Precompute and save RoPE
rope_path = os.path.join(COMPILE_DIR, "rope_embeddings.pt")
if os.path.exists(rope_path):
    rope_data = torch.load(rope_path, weights_only=True)
    rope_cos, rope_sin = rope_data["cos"], rope_data["sin"]
    print(f"RoPE loaded from cache: cos={rope_cos.shape}")
else:
    print("Precomputing RoPE...")
    rope_cos, rope_sin = precompute_flux_rope(transformer)
    torch.save({"cos": rope_cos, "sin": rope_sin}, rope_path)
    print(f"RoPE: cos={rope_cos.shape}, sin={rope_sin.shape}")

In [ ]:
# Part A: Embeddings + 8 MMDiT double blocks
print("=" * 60)
print("Part A: Embeddings + 8 MMDiT double blocks")
print("=" * 60)

part_a = FluxPartA(transformer)
part_a.eval()

part_a_inputs = (
    torch.randn(1, IMG_SEQ_LEN, 64, dtype=torch.bfloat16),
    torch.randn(1, TXT_SEQ_LEN, 4096, dtype=torch.bfloat16),
    torch.randn(1, 768, dtype=torch.bfloat16),
    torch.tensor([0.5], dtype=torch.bfloat16),
    torch.tensor([3.5], dtype=torch.bfloat16),
    rope_cos.contiguous(),
    rope_sin.contiguous(),
)

t_a = compile_part("transformer_part_a", part_a, part_a_inputs)
del part_a
gc.collect()

In [ ]:
# Part B: Single-stream blocks 0-18 (19 blocks)
print("=" * 60)
print("Part B: Single-stream blocks 0-18")
print("=" * 60)

single_blocks = transformer.single_transformer_blocks
part_b = FluxPartB(nn.ModuleList(list(single_blocks[:19])))
part_b.eval()

part_b_inputs = (
    torch.randn(1, TOTAL_SEQ_LEN, HIDDEN_DIM, dtype=torch.bfloat16),
    torch.randn(1, HIDDEN_DIM, dtype=torch.bfloat16),
    rope_cos.contiguous(),
    rope_sin.contiguous(),
)

t_b = compile_part("transformer_part_b", part_b, part_b_inputs)
del part_b
gc.collect()

In [ ]:
# Part C: Single-stream blocks 19-37 + norm_out + proj_out (19 blocks)
print("=" * 60)
print("Part C: Single-stream blocks 19-37 + output projection")
print("=" * 60)

part_c = FluxPartC(
    nn.ModuleList(list(single_blocks[19:])),
    transformer.norm_out,
    transformer.proj_out,
)
part_c.eval()

part_c_inputs = (
    torch.randn(1, TOTAL_SEQ_LEN, HIDDEN_DIM, dtype=torch.bfloat16),
    torch.randn(1, HIDDEN_DIM, dtype=torch.bfloat16),
    rope_cos.contiguous(),
    rope_sin.contiguous(),
)

t_c = compile_part("transformer_part_c", part_c, part_c_inputs)
del part_c, transformer
gc.collect()

print(f"\nTransformer total: {t_a + t_b + t_c:.1f}s ({(t_a + t_b + t_c) / 60:.1f} min)")

## 6. Compile VAE Decoder

The VAE decoder uses `--model-type=unet-inference` because `--auto-cast=matmult` causes compiler error 70 on SDK 2.29.

In [ ]:
# Reload pipeline for VAE if needed
try:
    _ = pipe.vae
except:
    print("Reloading pipeline for VAE...")
    pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)

vae_decoder = copy.deepcopy(pipe.vae.decoder)
vae_scaling = pipe.vae.config.scaling_factor
vae_shift = pipe.vae.config.shift_factor

# FLUX VAE has 16 latent channels
vae_input = torch.randn(1, 16, RESOLUTION // 8, RESOLUTION // 8, dtype=torch.bfloat16)
t_vae = compile_part("vae_decoder", vae_decoder, vae_input,
                      compiler_args=["--model-type=unet-inference"])

del vae_decoder, pipe
gc.collect()
print(f"\nVAE decoder: {t_vae:.1f}s")
print(f"\nAll 6 components compiled!")

## 7. Run Inference

Inference uses staged model loading:
1. **Stage 1**: Text encoding in a subprocess on core 0 (clean HBM)
2. **Stage 2**: Load transformer parts A/B/C on cores 1-3, run denoising loop
3. **Stage 3**: Unload transformer, load VAE decoder, decode latents to image

**Important**: The cell below sets `NEURON_RT_VISIBLE_CORES=1-3` for the main process. Core 0 is reserved for the text encoding subprocess.

In [ ]:
# Restrict main process to cores 1-3 (core 0 reserved for text encoder subprocess)
os.environ["NEURON_RT_VISIBLE_CORES"] = "1-3"

In [ ]:
def encode_text_subprocess(prompt, compile_dir, model_id):
    """Run CLIP + T5 text encoding in a subprocess with clean Neuron runtime on core 0.
    
    Returns (pooled_embeds, prompt_embeds) as CPU tensors.
    """
    script = f"""
import os
os.environ['NEURON_RT_VISIBLE_CORES'] = '0'

import torch
import torch_neuronx

root = '{compile_dir}'

# CLIP
from transformers import CLIPTokenizer
tokenizer = CLIPTokenizer.from_pretrained('{model_id}', subfolder='tokenizer')
clip = torch.jit.load(os.path.join(root, 'clip/model.pt'))
tokens = tokenizer('''{prompt}''', max_length={CLIP_SEQ_LEN}, padding='max_length',
                    truncation=True, return_tensors='pt')
pooled = clip(tokens.input_ids).cpu()
del clip

# T5
from transformers import T5TokenizerFast
tokenizer2 = T5TokenizerFast.from_pretrained('{model_id}', subfolder='tokenizer_2')
t5 = torch.jit.load(os.path.join(root, 't5/model.pt'))
tokens2 = tokenizer2('''{prompt}''', max_length={TXT_SEQ_LEN}, padding='max_length',
                      truncation=True, return_tensors='pt')
embeds = t5(tokens2.input_ids).cpu()
del t5

torch.save({{'pooled': pooled, 'embeds': embeds}}, '/tmp/flux_text_embeds.pt')
print(f'CLIP: {{pooled.shape}}, T5: {{embeds.shape}}')
print('TEXT_ENCODE_DONE')
"""
    script_path = "/tmp/flux_text_encode.py"
    with open(script_path, "w") as f:
        f.write(script)

    print("  Running text encoding in subprocess (core 0)...")
    result = subprocess.run(
        ["python3", script_path],
        capture_output=True, text=True, timeout=600,
    )
    print(f"  {result.stdout.strip()}")
    if result.returncode != 0:
        print(f"  stderr: {result.stderr[-500:]}")
        raise RuntimeError(f"Text encoding failed: {result.returncode}")

    data = torch.load("/tmp/flux_text_embeds.pt", weights_only=True)
    return data["pooled"], data["embeds"]

In [ ]:
# Stage 1: Text encoding
prompt = "a photo of an astronaut riding a horse on mars"
pooled_embeds, prompt_embeds = encode_text_subprocess(prompt, COMPILE_DIR, MODEL_ID)
print(f"CLIP pooled: {pooled_embeds.shape}, T5 embeds: {prompt_embeds.shape}")

In [ ]:
# Stage 2: Load transformer parts on cores 1-3
print("Loading transformer parts...")

traced_a = torch.jit.load(os.path.join(COMPILE_DIR, "transformer_part_a/model.pt"))
print("  Part A loaded")
traced_b = torch.jit.load(os.path.join(COMPILE_DIR, "transformer_part_b/model.pt"))
print("  Part B loaded")
traced_c = torch.jit.load(os.path.join(COMPILE_DIR, "transformer_part_c/model.pt"))
print("  Part C loaded")

# Load RoPE
rope_data = torch.load(os.path.join(COMPILE_DIR, "rope_embeddings.pt"), weights_only=True)
rope_cos = rope_data["cos"]
rope_sin = rope_data["sin"]

In [ ]:
from diffusers import FluxPipeline

# Load scheduler config (CPU only)
pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
scheduler = pipe.scheduler
vae_scaling = pipe.vae.config.scaling_factor
vae_shift = pipe.vae.config.shift_factor
del pipe
gc.collect()


def denoise(latents, scheduler, traced_a, traced_b, traced_c,
            prompt_embeds, pooled_embeds, guidance, rope_cos, rope_sin,
            num_steps, verbose=True):
    """Run the denoising loop through the 3-part split transformer."""
    # mu=1.15 for 1024x1024 with dynamic shifting
    scheduler.set_timesteps(num_steps, mu=1.15)
    for i, t in enumerate(scheduler.timesteps):
        timestep = (t / 1000).expand(1).to(torch.bfloat16)
        # Part A: embeddings + MMDiT blocks
        hidden_states, encoder_hidden_states, temb = traced_a(
            latents, prompt_embeds, pooled_embeds,
            timestep, guidance, rope_cos, rope_sin,
        )
        # Concatenate for single-stream processing
        combined = torch.cat([encoder_hidden_states, hidden_states], dim=1)
        # Part B: single blocks 0-18
        combined = traced_b(combined, temb, rope_cos, rope_sin)
        # Part C: single blocks 19-37 + output
        noise_pred = traced_c(combined, temb, rope_cos, rope_sin)
        # Scheduler step
        latents = scheduler.step(noise_pred, t, latents, return_dict=False)[0]
        if verbose and ((i + 1) % 7 == 0 or i == 0):
            print(f"    Step {i + 1}/{num_steps}")
    return latents


def make_latents():
    """Create and pack random latents for 1024x1024."""
    lat = torch.randn(1, 16, RESOLUTION // 8, RESOLUTION // 8, dtype=torch.bfloat16)
    B, C, H, W = lat.shape
    lat = lat.reshape(B, C, H // 2, 2, W // 2, 2)
    lat = lat.permute(0, 2, 4, 1, 3, 5).reshape(B, (H // 2) * (W // 2), C * 4)
    return lat, B, C, H, W

In [ ]:
# Generate image
guidance = torch.tensor([GUIDANCE_SCALE], dtype=torch.bfloat16)
latents, B, C, H, W = make_latents()

print(f"Generating image: {NUM_STEPS} steps, guidance={GUIDANCE_SCALE}")
t_start = time.time()
latents = denoise(
    latents, scheduler, traced_a, traced_b, traced_c,
    prompt_embeds, pooled_embeds, guidance, rope_cos, rope_sin,
    NUM_STEPS, verbose=True,
)
t_denoise = time.time() - t_start
print(f"\nDenoising: {t_denoise:.2f}s ({NUM_STEPS / t_denoise:.2f} steps/s)")

# Save latents for VAE decode
latents_for_vae = latents.clone()

In [ ]:
# Stage 3: Unload transformer, load VAE, decode
del traced_a, traced_b, traced_c
gc.collect()
print("Transformer parts unloaded")

traced_vae = torch.jit.load(os.path.join(COMPILE_DIR, "vae_decoder/model.pt"))
print("VAE decoder loaded")

# Unpack latents from sequence format back to spatial
latents = latents_for_vae
latents = latents.reshape(B, H // 2, W // 2, C, 2, 2)
latents = latents.permute(0, 3, 1, 4, 2, 5).reshape(B, C, H, W)
latents = (latents / vae_scaling) + vae_shift

t_vae = time.time()
image = traced_vae(latents)
t_vae = time.time() - t_vae
print(f"VAE decode: {t_vae:.2f}s")

# Convert to PIL
from PIL import Image

image = (image / 2 + 0.5).clamp(0, 1)
image = image.permute(0, 2, 3, 1).float().cpu().numpy()
img = Image.fromarray((image[0] * 255).astype(np.uint8))
img.save(os.path.join(COMPILE_DIR, "output.png"))
print(f"Image saved: {img.size}")
img

## 8. Benchmark

Benchmark the denoising loop only (text encoding is amortized, VAE is a single pass).

**Note**: To run the benchmark, the transformer NEFFs need to be reloaded. If you just ran inference above, restart the kernel or run the cells below in a fresh session with `NEURON_RT_VISIBLE_CORES=1-3` set.

In [ ]:
# Reload transformer parts if not loaded
# (This cell is safe to re-run; loading from NEFF is fast)
try:
    _ = traced_a
    print("Transformer parts already loaded")
except NameError:
    os.environ["NEURON_RT_VISIBLE_CORES"] = "1-3"
    import torch_neuronx  # re-import if needed
    
    traced_a = torch.jit.load(os.path.join(COMPILE_DIR, "transformer_part_a/model.pt"))
    traced_b = torch.jit.load(os.path.join(COMPILE_DIR, "transformer_part_b/model.pt"))
    traced_c = torch.jit.load(os.path.join(COMPILE_DIR, "transformer_part_c/model.pt"))
    rope_data = torch.load(os.path.join(COMPILE_DIR, "rope_embeddings.pt"), weights_only=True)
    rope_cos, rope_sin = rope_data["cos"], rope_data["sin"]
    
    from diffusers import FluxPipeline
    pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    scheduler = pipe.scheduler
    del pipe; gc.collect()
    print("Transformer parts reloaded")

In [ ]:
# Use cached text embeddings from earlier
guidance = torch.tensor([GUIDANCE_SCALE], dtype=torch.bfloat16)

N_BENCH = 5
print(f"Running benchmark: {N_BENCH} iterations (denoise only, {NUM_STEPS} steps each)\n")

latencies = []
for run in range(N_BENCH):
    lat, _, _, _, _ = make_latents()
    t0 = time.time()
    denoise(
        lat, scheduler, traced_a, traced_b, traced_c,
        prompt_embeds, pooled_embeds, guidance, rope_cos, rope_sin,
        NUM_STEPS, verbose=False,
    )
    elapsed = time.time() - t0
    latencies.append(elapsed)
    print(f"  Run {run + 1}: {elapsed:.2f}s")

latencies = np.array(latencies)
print(f"\n{'=' * 50}")
print(f"Mean denoise latency: {latencies.mean():.2f}s")
print(f"Std:                  {latencies.std():.2f}s")
print(f"Steps/sec:            {NUM_STEPS / latencies.mean():.2f}")
print(f"Min:                  {latencies.min():.2f}s")
print(f"Max:                  {latencies.max():.2f}s")

# Save results
results = {
    "model": "FLUX.1-lite-8B",
    "instance": "trn2.3xlarge",
    "sdk": "2.29",
    "resolution": f"{RESOLUTION}x{RESOLUTION}",
    "steps": NUM_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "split": "3-part (A: MMDiT, B: single 0-18, C: single 19-37+out)",
    "compiler_args": "-O1",
    "mean_denoise_latency_s": float(latencies.mean()),
    "std_s": float(latencies.std()),
    "steps_per_sec": float(NUM_STEPS / latencies.mean()),
}
with open(os.path.join(COMPILE_DIR, "benchmark_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {COMPILE_DIR}/benchmark_results.json")

## 9. Results Summary

### Performance

| Metric | Value |
|--------|-------|
| Mean denoise latency | ~39.0s |
| Steps/sec | ~0.72 |
| VAE decode | ~0.29s |
| Resolution | 1024x1024 |
| Denoising steps | 28 |
| Guidance scale | 3.5 |

### Compilation

| Component | Time | NEFF Size |
|-----------|------|-----------|
| CLIP | ~2 min | ~0.5 GB |
| T5-XXL | ~15 min | ~8 GB |
| Transformer Part A | ~20 min | ~9 GB |
| Transformer Part B | ~25 min | ~8 GB |
| Transformer Part C | ~25 min | ~8 GB |
| VAE Decoder | ~5 min | ~0.1 GB |
| **Total** | **~92 min** | **~33.6 GB** |

### Key technical findings

1. **Instruction limit**: The full 8B transformer has 8.6M instructions (limit: 5M). Splitting into 3 parts keeps each under 3M instructions.

2. **`--model-type=transformer` is harmful**: This flag replaces native softmax with a custom NKI kernel that is numerically wrong for FLUX attention. Use `-O1` only.

3. **HBM persistence**: The Neuron runtime doesn't release HBM on model delete within a process. Text encoding must use a separate subprocess.

4. **Scheduler `mu` parameter**: `FlowMatchEulerDiscreteScheduler` with `use_dynamic_shifting=True` requires `mu=1.15` for 1024x1024.

5. **Sequential bottleneck**: Each denoising step executes 3 sequential NEFFs (~1.39s/step), which is the primary latency bottleneck. TP across multiple cores could help but would require significant refactoring.